# Result Video -- Cascade on the 4 Held-Out Test Videos

The assignment PDF asks, among the presentation deliverables: "Varsa
sonuc videosu" (a result video, if available). This notebook produces
one: the cascade (Method 1b -- small-stem ResNet18 verifier + score
fusion, the best trained/deployable method at mAP@.5 0.786, see
`docs/decision_log.md`) run frame-by-frame on all 4 held-out test
videos, combined into a single annotated .mp4.

**Not a new method or a new evaluated run.** This calls the exact same
cascade logic as `scripts/25_cascade_score_fusion_eval.py` (same
checkpoints, same `sqrt(yolo_conf * verifier_prob)` fusion) -- the only
difference is it runs on every frame of the original videos instead of
the sampled frames used for scoring, and writes a video instead of a
COCO-format predictions file. Boxes are only drawn above the cascade's
own reported operating point (`draw_threshold: 0.38` in
`configs/36_render_result_video.yaml`) so what's shown matches the
numbers already reported in the presentation, not a lower threshold
that would look fuller on screen.

**GPU strongly recommended** -- at native `imgsz=1920`, YOLO alone runs
~7 FPS on a T4; on CPU this would take considerably longer over several
minutes of combined test footage.

**Before running:**
- Your verifier checkpoint
  (`results/cascade_verifier/resnet18_verifier_small_stem.pt`, produced
  by `notebooks/02_train_verifier.ipynb`) needs to be on Drive at
  `/content/drive/MyDrive/object-detection/resnet18_verifier_small_stem.pt`.
- A Kaggle API token (`kaggle.json`) to download the 4 raw test videos --
  same method the README documents for reproducing this project from
  scratch. If you already have the full `kmader/drone-videos` dataset on
  Drive from an earlier session, skip Step 1 and copy/unzip it into
  `data/raw/drone_videos/` instead.


## Step 0 -- Setup: clone the repo, install dependencies, confirm GPU

In [ ]:
import os
if not os.path.exists("object-detection-drone"):
    !git clone https://github.com/Kametor/object-detection-drone.git
%cd object-detection-drone
!git pull origin main --no-edit --no-rebase
!pip install -q -r requirements.txt


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow -- see the intro cell)")


## Step 1 -- Get the 4 raw test videos

Only the test-split videos are needed here (`data/splits/test.txt`), not
the full 13-video dataset -- downloading the whole Kaggle dataset works
too (matches the README), this just unzips a bit more than necessary.

In [ ]:
from google.colab import files
print("Upload your Kaggle API token (kaggle.json) -- from kaggle.com/settings -> API -> Create New Token")
uploaded = files.upload()
import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d kmader/drone-videos -p data/raw/ --unzip
!echo "test videos present:" && for f in "Berghouse Leopard Jog.mp4" "DJI_0501.MP4" "DJI_0596.MP4" "DJI_0862.MOV"; do ls -la "data/raw/drone_videos/$f" 2>/dev/null || echo "MISSING: $f"; done


## Step 2 -- Get the cascade's verifier checkpoint

The verifier is a custom-trained ResNet18 (`notebooks/02_train_verifier.ipynb`)
-- like all `.pt` checkpoints in this project, it's gitignored and lives
on Drive, not in the repo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("results/cascade_verifier", exist_ok=True)
VERIFIER_SRC = "/content/drive/MyDrive/object-detection/resnet18_verifier_small_stem.pt"
!cp "{VERIFIER_SRC}" results/cascade_verifier/resnet18_verifier_small_stem.pt
!ls -la results/cascade_verifier/resnet18_verifier_small_stem.pt


## Step 3 -- Render the video

Uses `configs/36_render_result_video.yaml` as committed (`device: cuda`,
`draw_threshold: 0.38`, all 4 test videos, every frame). `yolo26n.pt`
auto-downloads from Ultralytics if not already present on this VM.

In [ ]:
!python scripts/36_render_result_video.py --config configs/36_render_result_video.yaml


## Step 4 -- Preview it inline

In [ ]:
from IPython.display import HTML
from base64 import b64encode

video_path = "results/demo_video/cascade_test_videos.mp4"
print(f"{video_path}: {__import__('os').path.getsize(video_path) / 1e6:.1f} MB")
mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=720 controls><source src="{data_url}" type="video/mp4"></video>')


## Step 5 -- Save the video to Drive

Videos are gitignored (`*.mp4`) -- this is the deliverable's home, not a
git commit. Only the small manifest JSON goes to GitHub, in the next
step.

In [ ]:
import shutil
DRIVE_OUT = "/content/drive/MyDrive/object-detection/cascade_test_videos.mp4"
shutil.copy("results/demo_video/cascade_test_videos.mp4", DRIVE_OUT)
print(f"Saved to {DRIVE_OUT}")


## Step 6 -- Push the run manifest to GitHub

Not the video itself (gitignored, and it belongs on Drive per Step 5) --
just the small JSON record of this run (config hash, git SHA, frame
counts, timing), for reproducibility.

In [ ]:
import getpass
gh_token = getpass.getpass("GitHub personal access token: ")

!git config user.email "you@example.com"
!git config user.name "Colab"
!git add results/manifests
!git commit -m "Add result-video render manifest (cascade on the 4 test videos)"
!git pull origin main --no-edit --no-rebase
!git push https://{gh_token}@github.com/Kametor/object-detection-drone.git main
